# 06 — Routine / Habit Mining

Behavior-first follow-up after Stage 05c. No HOME/OFFICE relabeling.

Flow: frozen CP1 stays → per-stay local timezone → complete-link 200m behavior locations → support gate → daily sequences → directed OD routines → departure-time habits → split-half stability.


In [ ]:
from pathlib import Path
import importlib.util, os, subprocess, sys
import pandas as pd
import numpy as np

REPO_BRANCH=os.environ.get('GEOLIFE_REPO_BRANCH','main')
REPO_DIR=Path(os.environ.get('GEOLIFE_REPO_DIR','/tmp/geolife'))
VOLUME_ROOT=Path('/mnt/geolife-data')
CACHE_06=VOLUME_ROOT/'cache'/'06_routine_habit_mining'
CACHE_06.mkdir(parents=True,exist_ok=True)

def run(cmd,cwd=None): subprocess.run(cmd,cwd=cwd,check=True)
if (REPO_DIR/'.git').exists():
    current=subprocess.check_output(['git','-C',str(REPO_DIR),'branch','--show-current'],text=True).strip()
    if current!=REPO_BRANCH: run(['git','-C',str(REPO_DIR),'checkout',REPO_BRANCH])
else:
    run(['git','clone','--depth','1','--branch',REPO_BRANCH,'https://github.com/tanh1c/geolife.git',str(REPO_DIR)])
for p in (REPO_DIR,REPO_DIR/'src'):
    if str(p) not in sys.path: sys.path.insert(0,str(p))
try:
    from geolife.model import HomeOfficeConfig
except ModuleNotFoundError:
    run([sys.executable,'-m','pip','install','-q','-e','.'],cwd=REPO_DIR)
def load_module(path,name):
    spec=importlib.util.spec_from_file_location(name,path); mod=importlib.util.module_from_spec(spec); sys.modules[name]=mod; spec.loader.exec_module(mod); return mod
behavior=load_module(REPO_DIR/'analysis'/'03a_user_behavior_deep_dive.py','behavior03a')
stage06=load_module(REPO_DIR/'analysis'/'06_routine_habit_mining.py','stage06')
display(pd.DataFrame([stage06.synthetic_self_check()]))


## 1. Frozen representation parity

Stage 06 uses the broader behavior-EDA scope, not the 97-user Beijing semantic cohort. Expected anchors: 104 users with at least one recurring complete-link 200m location.


In [ ]:
def find_cache(name):
    for p in [VOLUME_ROOT/'cache'/'03a_user_behavior_deep_dive'/name,VOLUME_ROOT/'artifacts'/'03a'/name,REPO_DIR/'artifacts'/'03a'/name]:
        if p.exists(): return p
    hits=sorted(VOLUME_ROOT.glob(f'**/{name}'))
    if hits: return hits[0]
    raise FileNotFoundError(name)
stays=pd.read_pickle(find_cache('stays_baseline_v1.pkl'))
point_days=pd.read_pickle(find_cache('cleaned_point_daily_metrics.pkl'))
assert len(stays)==5821 and stays['user_id'].nunique()==136
resolved=behavior.resolve_stay_timezones(stays)
resolved=resolved.loc[resolved['timezone_id'].notna()].copy()
clustered,location_summary=behavior.cluster_behavior_locations(resolved,200.0)
recurring_users=set(location_summary.loc[location_summary['stay_count'].ge(2),'user_id'].astype(str))
assert len(recurring_users)==104
display(pd.DataFrame([{'cp1_stays':len(stays),'stay_users':stays['user_id'].nunique(),'resolved_users':resolved['user_id'].nunique(),'behavior_locations':len(location_summary),'recurring_location_users':len(recurring_users),'recurring_locations':int(location_summary['stay_count'].ge(2).sum())}]))


## 2. Coverage gate and routine mining

Only days passing the existing `usable_for_motif` support gate contribute to routine evidence. Consecutive duplicate locations are collapsed before directed OD transitions are created.


In [ ]:
behavior_points=behavior._point_days_for_behavior(point_days)
daily_support=behavior.build_user_day_features(clustered,behavior_points)
audit06=stage06.run_audit(clustered,daily_support)
tables=stage06.aggregate_report_tables(audit06.edge_summary,audit06.user_summary,audit06.split_half)
display(tables['coverage'])
display(tables['departure_time_modes'])
display(tables['split_half_stability'])


## 3. Sensitivity

Sweep repeat-day thresholds 2/3/5 and departure circular concentration 0.3/0.5/0.7. These are descriptive support checks, not semantic classifiers.


In [ ]:
display(audit06.sensitivity)
repeated=audit06.edge_summary.loc[audit06.edge_summary['active_days'].ge(2)]
display(pd.DataFrame([{'repeated_edges':len(repeated),'median_edge_day_share':float(repeated['edge_day_share'].median()) if len(repeated) else np.nan,'median_departure_concentration':float(repeated['departure_concentration'].median()) if len(repeated) else np.nan,'multimodal_time_edges':int(pd.to_numeric(repeated['time_mode_count'],errors='coerce').ge(2).sum())}]))


## 4. Save private and aggregate outputs

User-level sequences/edges remain on the private Modal Volume. Only aggregate summaries should be copied into reports.


In [ ]:
audit06.day_sequences.to_pickle(CACHE_06/'day_sequences_private.pkl')
audit06.transitions.to_pickle(CACHE_06/'transitions_private.pkl')
audit06.edge_summary.to_pickle(CACHE_06/'edge_summary_private.pkl')
audit06.user_summary.to_pickle(CACHE_06/'user_summary_private.pkl')
audit06.split_half.to_pickle(CACHE_06/'split_half_private.pkl')
for name,frame in tables.items(): frame.to_csv(CACHE_06/f'{name}.csv',index=False)
audit06.sensitivity.to_csv(CACHE_06/'routine_sensitivity.csv',index=False)
print('Saved:',CACHE_06)


## Decision gate

Proceed to Stage 07 change detection only if routine coverage and split-half stability are sufficient. Coverage must be gated before interpreting changes; sparse observation is not behavioral change.
